In [1]:
import os
os.environ['PYTHONSAFEPATH'] = '1'
from sklearn.datasets import fetch_20newsgroups
import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))


newsgroup = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))
documents = newsgroup.data

print(f"Number of documents = {len(documents)}")
print(documents[0][:500])

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words and len(t) >= 3]
    return tokens

p_docs = [preprocess(doc) for doc in documents]  # p_docs = processed docs
print(p_docs[0][:20])

avg_length = sum(len(doc) for doc in p_docs)/len(p_docs)
print("Average Document Length: ", avg_length)

docs = [doc for doc in p_docs if len(doc)>0]
print(f"Number of documents after cleaning: {len(docs)}")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/requested/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Number of documents = 18846


I am sure some bashers of Pens fans are pretty confused about the lack
of any kind of posts about the recent Pens massacre of the Devils. Actually,
I am  bit puzzled too and a bit relieved. However, I am going to put an end
to non-PIttsburghers' relief with a bit of praise for the Pens. Man, they
are killing those Devils worse than I thought. Jagr just showed you why
he is much better than his regular season stats. He is also a lot
fo fun to watch in the playoffs. Bowman should let JAgr have a
['sure', 'bashers', 'pens', 'fans', 'pretty', 'confused', 'lack', 'kind', 'posts', 'recent', 'pens', 'massacre', 'devils', 'actually', 'bit', 'puzzled', 'bit', 'relieved', 'however', 'going']
Average Document Length:  93.68990767271569
Number of documents after cleaning: 18295


Buliding Vocabulary:

In [2]:
from collections import Counter

word_counts = Counter(word for doc in docs for word in doc)

print(f"Most common words are: {word_counts.most_common(10)}")
print(f"Least common words are: {word_counts.most_common()[-10:]}")
print(f"Total unique words are: {len(word_counts)}")

Most common words are: [('one', 10409), ('would', 10191), ('like', 6529), ('people', 6459), ('get', 5883), ('know', 5774), ('also', 5592), ('use', 5033), ('think', 5005), ('time', 4879)]
Least common words are: [('sured', 1), ('intracranial', 1), ('paroxysmal', 1), ('hemicrania', 1), ('aneurysm', 1), ('recepticles', 1), ('equi', 1), ('fcom', 1), ('pontaic', 1), ('definate', 1)]
Total unique words are: 93467


In [3]:
min_frequency = 4
vocabulary = {word for word, count in word_counts.items() if count >= min_frequency}
print(f"Vocabulary size after filtering: {len(vocabulary)}")
print(vocabulary)

Vocabulary size after filtering: 29772
{'kid', 'afterall', 'ojala', 'mqa', 'die', 'dialogue', 'hunters', 'mget', 'proportional', 'haiastan', 'lexicons', 'pius', 'rejects', 'rix', 'hoy', 'borschevsky', 'semite', 'propelled', 'falcon', 'styled', 'unexplainable', 'juggernaut', 'readers', 'redesigned', 'accsys', 'insult', 'assumption', 'mains', 'filtration', 'synchronization', 'combine', 'eece', 'patterns', 'engin', 'individual', 'keep', 'narrative', 'snijpunten', 'tangible', 'eng', 'unprovable', 'ioy', 'voters', 'auction', 'rebuilding', 'finder', 'wmhints', 'usual', 'wonders', 'backon', 'yeh', 'protectors', 'usno', 'yildiz', 'stdout', 'clinical', 'suspected', 'indeed', 'hills', 'nhtsa', 'silk', 'apparatus', 'imaginable', 'ecl', 'rationalizations', 'retreat', 'projectors', 'completes', 'hahn', 'boom', 'imageheight', 'opposed', 'accusing', 'attractions', 'explores', 'owens', 'rkba', 'blasphemy', 'planner', 'drawer', 'repeater', 'features', 'prelude', 'taped', 'multiply', 'frameworks', 'spea

word2idx and idx2word mappings:

In [4]:
word2idx = {word: idx for idx, word in enumerate(sorted(vocabulary))}
idx2word = {idx: word for word, idx in word2idx.items()}

In [5]:
# Generating pairs for training for word2vec
def generate_training_pairs(docs, word2idx, window_size=2):
    pairs = []
    for doc in docs:
        doc = [word for word in doc if word in word2idx]

        for i, target_word in enumerate(doc):
            target_idx = word2idx[target_word]

            left = max(0, i-window_size)
            right = min(len(doc), i+window_size+1)

            for j in range(left, right):
                if i == j:
                    continue
                context_word = doc[j]
                context_idx = word2idx[context_word]
                pairs.append((target_idx, context_idx))

    return pairs

In [6]:
pairs = generate_training_pairs(docs, word2idx, window_size=2)

print(f"Total training pairs = {len(pairs)}")
print(f"First 5 pairs: {pairs[:5]}")

for target_idx, context_idx in pairs[:5]:
    print(f"{idx2word[target_idx]} -> {idx2word[context_idx]}")

Total training pairs = 6594196
First 5 pairs: [(25538, 2249), (25538, 19240), (2249, 25538), (2249, 19240), (2249, 9369)]
sure -> bashers
sure -> pens
bashers -> sure
bashers -> pens
bashers -> fans


In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "mps" if torch.backends.mps.is_available() else "cpu"
vocab_size, embed_dim, num_neg = len(word2idx), 100, 5

In [8]:


counts = torch.tensor([word_counts[idx2word[i]] for i in range(vocab_size)], dtype=torch.float)
noise_dist = (counts ** 0.75 / (counts ** 0.75).sum()).to(device)

class SkipGramNS(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.in_embed = nn.Embedding(vocab_size, dim)  # target vectors
        self.out_embed = nn.Embedding(vocab_size, dim) # context vectors
        nn.init.uniform_(self.in_embed.weight, -0.5 / dim, 0.5 / dim)
        nn.init.zeros_(self.out_embed.weight)
    
    def forward(self, target, context, negatives):
        v = self.in_embed(target)
        pos = (v * self.out_embed(context)).sum(1)
        neg = torch.bmm(self.out_embed(negatives), v.unsqueeze(2)).squeeze(2)
        return -(F.logsigmoid(pos) + F.logsigmoid(-neg).sum(1)).mean()

model = SkipGramNS(vocab_size, embed_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
pairs_t = torch.tensor(pairs, dtype=torch.long)
batch_size = 1024

for epoch in range(3):
    perm = torch.randperm(len(pairs_t))
    total = 0.0
    for i in range(0, len(perm), batch_size):
        batch = pairs_t[perm[i:i+batch_size]].to(device)
        target, context = batch[:, 0], batch[:, 1]
        negatives = torch.multinomial(noise_dist, len(batch) * num_neg, replacement=True).view(len(batch), num_neg)
        loss = model(target, context, negatives)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()
    print(f" epoch {epoch}: loss = {total/(len(perm) // batch_size):.4f}")

emb = F.normalize(model.in_embed.weight.detach(), dim=1)
def most_similar(word, k=10):
    sims = emb @ emb[word2idx[word]]
    return [(idx2word[i], round(sims[i].item(), 3)) for i in sims.topk(k+1).indices.tolist()[1:]]



 epoch 0: loss = 2.5475
 epoch 1: loss = 2.2171
 epoch 2: loss = 2.0961


In [9]:
for w in ["god", "car", "space", "windows"]:
    print(w, most_similar(w))

god [('spirit', 0.807), ('salvation', 0.778), ('deity', 0.774), ('spirits', 0.765), ('subsistence', 0.762), ('jokke', 0.761), ('satan', 0.76), ('eternal', 0.759), ('triune', 0.759), ('omniscience', 0.759)]
car [('honda', 0.777), ('sporty', 0.776), ('bike', 0.769), ('tires', 0.764), ('cars', 0.763), ('toyota', 0.76), ('celica', 0.755), ('volvo', 0.755), ('dealership', 0.755), ('rebuilt', 0.751)]
space [('shuttle', 0.724), ('hubble', 0.671), ('shuttles', 0.652), ('ksc', 0.64), ('mir', 0.638), ('launch', 0.631), ('sts', 0.63), ('payloads', 0.627), ('manned', 0.621), ('colony', 0.617)]
windows [('workgroups', 0.849), ('excel', 0.817), ('microsoft', 0.808), ('desktop', 0.803), ('apps', 0.803), ('desqview', 0.787), ('helpfiles', 0.786), ('excursion', 0.785), ('xvision', 0.774), ('procomm', 0.77)]
